In [1]:
import sys
sys.path.insert(0, '../..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
import time
import json

sns.set_theme(style="whitegrid")
Path('../../data/processed/plots').mkdir(
    parents=True, exist_ok=True)

PROC = '../../data/processed/'
FEAT = '../../data/features/'

print("✅ Imports ready")

✅ Imports ready


In [2]:
ratings = pd.read_csv(PROC + 'ratings_cleaned.csv')
movies  = pd.read_csv(PROC + 'movies_master.csv',
                      low_memory=False)

# Keep only columns we need
movies = movies[['movieId', 'id', 'title']].dropna(
    subset=['movieId'])
movies['movieId'] = movies['movieId'].astype(int)

print(f"Ratings : {ratings.shape}")
print(f"Movies  : {movies.shape}")
print(f"\nRatings sample:")
print(ratings.head())

Ratings : (100004, 4)
Movies  : (45454, 3)

Ratings sample:
   userId  movieId  rating            timestamp
0       1       31     2.5  2009-12-14 02:52:24
1       1     1029     3.0  2009-12-14 02:52:59
2       1     1061     3.0  2009-12-14 02:53:02
3       1     1129     2.0  2009-12-14 02:53:05
4       1     1172     4.0  2009-12-14 02:53:25


In [3]:
## Train/Test Split

# Sort by timestamp — temporal split
ratings = ratings.sort_values('timestamp').reset_index(
    drop=True)

# 80% train, 10% val, 10% test
n         = len(ratings)
train_end = int(n * 0.80)
val_end   = int(n * 0.90)

train = ratings.iloc[:train_end].copy()
val   = ratings.iloc[train_end:val_end].copy()
test  = ratings.iloc[val_end:].copy()

print(f"Train : {len(train):,} ratings "
      f"({len(train)/n*100:.0f}%)")
print(f"Val   : {len(val):,}   ratings "
      f"({len(val)/n*100:.0f}%)")
print(f"Test  : {len(test):,}  ratings "
      f"({len(test)/n*100:.0f}%)")

# Only evaluate on users/movies seen in training
train_users  = set(train['userId'].unique())
train_movies = set(train['movieId'].unique())

test_clean = test[
    test['userId'].isin(train_users) &
    test['movieId'].isin(train_movies)
].copy()

print(f"\nTest after cold-start filter: "
      f"{len(test_clean):,} ratings")
print(f"Cold-start removed          : "
      f"{len(test) - len(test_clean):,} ratings")
print(f"→ These are the cold-start problem cases")

Train : 80,003 ratings (80%)
Val   : 10,000   ratings (10%)
Test  : 10,001  ratings (10%)

Test after cold-start filter: 424 ratings
Cold-start removed          : 9,577 ratings
→ These are the cold-start problem cases


In [4]:
# Build User-Item Matrix

import joblib
import scipy.sparse as sp

# Build from training data only
user_ids  = sorted(train['userId'].unique())
movie_ids = sorted(train['movieId'].unique())

user2idx  = {u: i for i, u in enumerate(user_ids)}
movie2idx = {m: i for i, m in enumerate(movie_ids)}
idx2user  = {i: u for u, i in user2idx.items()}
idx2movie = {i: m for m, i in movie2idx.items()}

row = train['userId'].map(user2idx).values
col = train['movieId'].map(movie2idx).values
val = train['rating'].values

# Sparse matrix — users × movies
train_matrix = sp.csr_matrix(
    (val, (row, col)),
    shape=(len(user_ids), len(movie_ids))
)

sparsity = 1 - (train_matrix.nnz /
                (train_matrix.shape[0] *
                 train_matrix.shape[1]))

print(f"User-Item matrix shape : {train_matrix.shape}")
print(f"Non-zero entries       : {train_matrix.nnz:,}")
print(f"Sparsity               : {sparsity*100:.4f}%")
print(f"\nThis extreme sparsity is WHY CF struggles")
print(f"Most users have rated < 0.1% of all movies")

User-Item matrix shape : (547, 7356)
Non-zero entries       : 80,003
Sparsity               : 98.0117%

This extreme sparsity is WHY CF struggles
Most users have rated < 0.1% of all movies


In [5]:
## Cosine Similarity from scratch

def cosine_similarity_sparse(matrix):
    """
    Compute cosine similarity for all row pairs.
    Works on sparse matrix efficiently.

    similarity(u,v) = dot(u,v) / (norm(u) * norm(v))
    """
    # Normalise each row to unit length
    norms = np.sqrt(np.array(
        matrix.multiply(matrix).sum(axis=1)
    )).flatten()

    # Avoid division by zero
    norms[norms == 0] = 1e-10

    # Normalise
    norm_matrix = matrix.multiply(
        1.0 / norms[:, np.newaxis])

    # Cosine similarity = normalised dot product
    similarity = (norm_matrix @ norm_matrix.T).toarray()

    return similarity


def pearson_similarity(matrix_dense, min_common=3):
    """
    Compute Pearson correlation between users.
    Corrects for individual rating bias (generous vs harsh raters).
    min_common: minimum co-rated items required
    """
    n_users = matrix_dense.shape[0]
    sim     = np.zeros((n_users, n_users))

    for i in range(n_users):
        for j in range(i, n_users):
            # Find commonly rated items
            mask = (matrix_dense[i] > 0) & \
                   (matrix_dense[j] > 0)
            common = mask.sum()

            if common < min_common:
                sim[i][j] = sim[j][i] = 0
                continue

            u = matrix_dense[i][mask]
            v = matrix_dense[j][mask]

            # Mean-centre ratings
            u_c = u - u.mean()
            v_c = v - v.mean()

            denom = (np.sqrt((u_c**2).sum()) *
                     np.sqrt((v_c**2).sum()))

            if denom < 1e-10:
                sim[i][j] = sim[j][i] = 0
            else:
                r = (u_c * v_c).sum() / denom
                sim[i][j] = sim[j][i] = r

    return sim


print("✅ Similarity functions defined")
print("""
Cosine similarity  → fast, works on sparse matrix
Pearson similarity → slower, corrects for rating bias
                     only practical on small matrices
""")

✅ Similarity functions defined

Cosine similarity  → fast, works on sparse matrix
Pearson similarity → slower, corrects for rating bias
                     only practical on small matrices



In [6]:
## User-CF Recommender

class UserCF:
    """
    User-Based Collaborative Filtering
    Built from scratch — no libraries
    """

    def __init__(self, top_k_users: int = 20):
        self.top_k_users  = top_k_users
        self.train_matrix = None
        self.user_sim     = None
        self.user2idx     = None
        self.movie2idx    = None
        self.idx2movie    = None
        self.user_ids     = None
        self.movie_ids    = None

    def fit(self, matrix, user2idx, movie2idx,
            idx2movie, user_ids, movie_ids):
        self.train_matrix = matrix
        self.user2idx     = user2idx
        self.movie2idx    = movie2idx
        self.idx2movie    = idx2movie
        self.user_ids     = user_ids
        self.movie_ids    = movie_ids

        print("Computing user-user cosine similarity...")
        start = time.time()
        self.user_sim = cosine_similarity_sparse(matrix)
        elapsed = time.time() - start
        print(f"✅ Similarity matrix: "
              f"{self.user_sim.shape} "
              f"in {elapsed:.1f}s")

    def recommend(self, user_id: int,
                  n_recommendations: int = 10) -> list:
        """
        Generate top-N recommendations for a user.
        Returns list of (movieId, predicted_score)
        """
        if user_id not in self.user2idx:
            return []  # cold start — no history

        u_idx = self.user2idx[user_id]

        # Get top-K similar users (exclude self)
        sim_scores = self.user_sim[u_idx].copy()
        sim_scores[u_idx] = 0  # exclude self

        top_k_idx = np.argsort(sim_scores)[::-1][
            :self.top_k_users]

        # Movies already rated by target user
        rated_mask = self.train_matrix[u_idx].toarray()\
                         .flatten() > 0

        # Aggregate scores from similar users
        scores = np.zeros(self.train_matrix.shape[1])

        for k_idx in top_k_idx:
            sim = sim_scores[k_idx]
            if sim <= 0:
                continue
            neighbor_ratings = self.train_matrix[
                k_idx].toarray().flatten()
            scores += sim * neighbor_ratings

        # Zero out already-rated movies
        scores[rated_mask] = 0

        # Top-N movies by score
        top_movie_idx = np.argsort(scores)[::-1][
            :n_recommendations]

        recommendations = [
            (self.idx2movie[idx], float(scores[idx]))
            for idx in top_movie_idx
            if scores[idx] > 0
        ]

        return recommendations

    def predict_rating(self, user_id: int,
                       movie_id: int) -> float:
        """Predict rating for a specific user-movie pair"""
        if user_id not in self.user2idx or \
           movie_id not in self.movie2idx:
            return 0.0

        u_idx = self.user2idx[user_id]
        m_idx = self.movie2idx[movie_id]

        sim_scores = self.user_sim[u_idx].copy()
        sim_scores[u_idx] = 0

        top_k_idx = np.argsort(sim_scores)[::-1][
            :self.top_k_users]

        numerator   = 0.0
        denominator = 0.0

        for k_idx in top_k_idx:
            sim    = sim_scores[k_idx]
            rating = self.train_matrix[k_idx, m_idx]
            if sim > 0 and rating > 0:
                numerator   += sim * rating
                denominator += abs(sim)

        if denominator == 0:
            return 0.0
        return numerator / denominator


print("✅ UserCF class defined")

✅ UserCF class defined


In [7]:
# Item-CF Recommender

class ItemCF:
    """
    Item-Based Collaborative Filtering
    Built from scratch — no libraries
    """

    def __init__(self, top_k_items: int = 20):
        self.top_k_items  = top_k_items
        self.train_matrix = None
        self.item_sim     = None
        self.user2idx     = None
        self.movie2idx    = None
        self.idx2movie    = None

    def fit(self, matrix, user2idx, movie2idx,
            idx2movie, user_ids, movie_ids):
        self.train_matrix = matrix
        self.user2idx     = user2idx
        self.movie2idx    = movie2idx
        self.idx2movie    = idx2movie

        print("Computing item-item cosine similarity...")
        print(f"Matrix: {matrix.T.shape} "
              f"(items × users)")
        start = time.time()

        # Transpose — now items × users
        self.item_sim = cosine_similarity_sparse(
            matrix.T)

        elapsed = time.time() - start
        print(f"✅ Item similarity matrix: "
              f"{self.item_sim.shape} "
              f"in {elapsed:.1f}s")

    def recommend(self, user_id: int,
                  n_recommendations: int = 10) -> list:
        """Generate recommendations based on item similarity"""
        if user_id not in self.user2idx:
            return []

        u_idx = self.user2idx[user_id]

        # Movies rated by this user
        user_ratings = self.train_matrix[
            u_idx].toarray().flatten()
        rated_indices = np.where(user_ratings > 0)[0]

        if len(rated_indices) == 0:
            return []

        # For each unrated movie — score based on
        # similarity to user's rated movies
        scores = np.zeros(self.train_matrix.shape[1])

        for m_idx in rated_indices:
            rating   = user_ratings[m_idx]
            sim_row  = self.item_sim[m_idx]
            scores  += rating * sim_row

        # Zero out already-rated
        scores[user_ratings > 0] = 0

        top_movie_idx = np.argsort(scores)[::-1][
            :n_recommendations]

        return [
            (self.idx2movie[idx], float(scores[idx]))
            for idx in top_movie_idx
            if scores[idx] > 0
        ]

    def predict_rating(self, user_id: int,
                       movie_id: int) -> float:
        """Predict rating using item similarity"""
        if user_id not in self.user2idx or \
           movie_id not in self.movie2idx:
            return 0.0

        u_idx = self.user2idx[user_id]
        m_idx = self.movie2idx[movie_id]

        user_ratings  = self.train_matrix[
            u_idx].toarray().flatten()
        rated_indices = np.where(user_ratings > 0)[0]

        sim_scores = self.item_sim[m_idx][rated_indices]
        ratings    = user_ratings[rated_indices]

        top_k = np.argsort(sim_scores)[::-1][
            :self.top_k_items]
        sim_k  = sim_scores[top_k]
        rate_k = ratings[top_k]

        denom = np.abs(sim_k).sum()
        if denom == 0:
            return 0.0
        return (sim_k * rate_k).sum() / denom


print("✅ ItemCF class defined")

✅ ItemCF class defined


In [8]:
## Train Both Models

# Train User-CF
print("Training User-CF...")
user_cf = UserCF(top_k_users=20)
user_cf.fit(
    train_matrix, user2idx, movie2idx,
    idx2movie, user_ids, movie_ids
)

# Train Item-CF
print("\nTraining Item-CF...")
item_cf = ItemCF(top_k_items=20)
item_cf.fit(
    train_matrix, user2idx, movie2idx,
    idx2movie, user_ids, movie_ids
)

print("\n✅ Both models trained")

Training User-CF...
Computing user-user cosine similarity...
✅ Similarity matrix: (547, 547) in 0.0s

Training Item-CF...
Computing item-item cosine similarity...
Matrix: (7356, 547) (items × users)
✅ Item similarity matrix: (7356, 7356) in 0.4s

✅ Both models trained


In [9]:
# Generate Sample Recommendations

# Pick a test user
sample_user = train['userId'].value_counts().index[0]

print(f"Generating recommendations for User {sample_user}")
print("=" * 55)

# Movies this user already rated
user_history = train[train['userId'] == sample_user]\
    .merge(movies, on='movieId')\
    .sort_values('rating', ascending=False)

print(f"\nUser's top 5 rated movies:")
print(user_history[['title', 'rating']].head())

# User-CF recommendations
print(f"\nUser-CF recommendations:")
user_cf_recs = user_cf.recommend(sample_user, 10)
for movie_id, score in user_cf_recs[:5]:
    title = movies[movies['movieId'] == movie_id][
        'title'].values
    title = title[0] if len(title) > 0 else str(movie_id)
    print(f"  {title:<45} score: {score:.3f}")

# Item-CF recommendations
print(f"\nItem-CF recommendations:")
item_cf_recs = item_cf.recommend(sample_user, 10)
for movie_id, score in item_cf_recs[:5]:
    title = movies[movies['movieId'] == movie_id][
        'title'].values
    title = title[0] if len(title) > 0 else str(movie_id)
    print(f"  {title:<45} score: {score:.3f}")

Generating recommendations for User 547

User's top 5 rated movies:
                     title  rating
0       North by Northwest     5.0
154          Jules and Jim     5.0
157             GoodFellas     5.0
158        Ordinary People     5.0
159  The Last Picture Show     5.0

User-CF recommendations:
  Star Wars                                     score: 29.967
  The Princess Bride                            score: 27.510
  Saving Private Ryan                           score: 27.336
  Monty Python and the Holy Grail               score: 25.080
  Patton                                        score: 23.658

Item-CF recommendations:
  Saving Private Ryan                           score: 1091.513
  Blazing Saddles                               score: 1077.874
  Monty Python and the Holy Grail               score: 1072.850
  Patton                                        score: 1072.115
  Malcolm X                                     score: 1047.883


In [10]:
## Evaluation Metrics

def rmse(predictions: list) -> float:
    """Root Mean Square Error"""
    if not predictions:
        return float('inf')
    errors = [(true - pred)**2
              for true, pred in predictions]
    return np.sqrt(np.mean(errors))

def mae(predictions: list) -> float:
    """Mean Absolute Error"""
    if not predictions:
        return float('inf')
    errors = [abs(true - pred)
              for true, pred in predictions]
    return np.mean(errors)

def precision_at_k(recommended: list,
                   relevant: set,
                   k: int = 10) -> float:
    """Precision@K — of top K, how many are relevant"""
    top_k = recommended[:k]
    hits  = sum(1 for m in top_k if m in relevant)
    return hits / k if k > 0 else 0.0

def recall_at_k(recommended: list,
                relevant: set,
                k: int = 10) -> float:
    """Recall@K — of all relevant, how many in top K"""
    if not relevant:
        return 0.0
    top_k = recommended[:k]
    hits  = sum(1 for m in top_k if m in relevant)
    return hits / len(relevant)

def ndcg_at_k(recommended: list,
              relevant: set,
              k: int = 10) -> float:
    """NDCG@K — ranking quality (order matters)"""
    top_k = recommended[:k]
    dcg   = sum(
        1.0 / np.log2(i + 2)
        for i, m in enumerate(top_k)
        if m in relevant
    )
    # Ideal DCG
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 2)
               for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

print("✅ Evaluation metrics defined")
print("""
RMSE        → rating prediction error
MAE         → average absolute error
Precision@K → of top K shown, how many relevant?
Recall@K    → of all relevant, how many in top K?
NDCG@K      → ranking quality — order matters
""")

✅ Evaluation metrics defined

RMSE        → rating prediction error
MAE         → average absolute error
Precision@K → of top K shown, how many relevant?
Recall@K    → of all relevant, how many in top K?
NDCG@K      → ranking quality — order matters



In [11]:
#Evaluated Both Models

def evaluate_model(model, test_df, movies_df,
                   model_name: str,
                   n_users: int = 200,
                   k: int = 10) -> dict:
    """
    Evaluate a CF model on test set.
    n_users: evaluate on sample for speed
    """
    print(f"\nEvaluating {model_name}...")

    # Rating prediction metrics (RMSE, MAE)
    predictions = []
    test_sample = test_df.sample(
        min(5000, len(test_df)),
        random_state=42
    )

    for _, row in test_sample.iterrows():
        pred = model.predict_rating(
            int(row['userId']),
            int(row['movieId'])
        )
        if pred > 0:
            predictions.append((row['rating'], pred))

    rmse_score = rmse(predictions)
    mae_score  = mae(predictions)

    # Ranking metrics (Precision, Recall, NDCG)
    precisions, recalls, ndcgs = [], [], []

    test_users = test_df['userId'].unique()
    eval_users = np.random.choice(
        test_users,
        size=min(n_users, len(test_users)),
        replace=False
    )

    for user_id in eval_users:
        # Relevant movies = rated >= 4.0 in test
        relevant = set(
            test_df[
                (test_df['userId'] == user_id) &
                (test_df['rating'] >= 4.0)
            ]['movieId'].values
        )
        if not relevant:
            continue

        # Get recommendations
        recs = model.recommend(int(user_id), k)
        rec_movies = [m for m, _ in recs]

        precisions.append(
            precision_at_k(rec_movies, relevant, k))
        recalls.append(
            recall_at_k(rec_movies, relevant, k))
        ndcgs.append(
            ndcg_at_k(rec_movies, relevant, k))

    results = {
        "model":        model_name,
        "rmse":         round(rmse_score, 4),
        "mae":          round(mae_score,  4),
        f"precision@{k}": round(
            np.mean(precisions), 4),
        f"recall@{k}":    round(
            np.mean(recalls), 4),
        f"ndcg@{k}":      round(
            np.mean(ndcgs), 4),
        "n_predictions":  len(predictions),
        "n_users_eval":   len(precisions),
    }

    print(f"  RMSE          : {results['rmse']}")
    print(f"  MAE           : {results['mae']}")
    print(f"  Precision@{k}  : "
          f"{results[f'precision@{k}']}")
    print(f"  Recall@{k}     : "
          f"{results[f'recall@{k}']}")
    print(f"  NDCG@{k}       : "
          f"{results[f'ndcg@{k}']}")

    return results


# Evaluate both models
results_user_cf = evaluate_model(
    user_cf, test_clean, movies, "User-CF")
results_item_cf = evaluate_model(
    item_cf, test_clean, movies, "Item-CF")


Evaluating User-CF...
  RMSE          : 1.0497
  MAE           : 0.7457
  Precision@10  : 0.0714
  Recall@10     : 0.0543
  NDCG@10       : 0.0686

Evaluating Item-CF...
  RMSE          : 0.8904
  MAE           : 0.6438
  Precision@10  : 0.0286
  Recall@10     : 0.0248
  NDCG@10       : 0.0367


In [12]:
# Result Comparison Table
results_df = pd.DataFrame([
    results_user_cf,
    results_item_cf
])

print("\nCF BASELINE RESULTS")
print("=" * 60)
print(results_df[[
    'model', 'rmse', 'mae',
    'precision@10', 'recall@10', 'ndcg@10'
]].to_string(index=False))

print("""
INTERPRETATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
These are your BASELINE numbers.
Every model you build in Days 9–14 must beat these.

Low Precision@10 and NDCG@10 are EXPECTED here.
This is the evidence your report uses to justify
moving to modern generative retrieval models.
""")



CF BASELINE RESULTS
  model   rmse    mae  precision@10  recall@10  ndcg@10
User-CF 1.0497 0.7457        0.0714     0.0543   0.0686
Item-CF 0.8904 0.6438        0.0286     0.0248   0.0367

INTERPRETATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
These are your BASELINE numbers.
Every model you build in Days 9–14 must beat these.

Low Precision@10 and NDCG@10 are EXPECTED here.
This is the evidence your report uses to justify
moving to modern generative retrieval models.



In [13]:
# CF Failure Analysis

print("CF FAILURE ANALYSIS")
print("=" * 55)

# 1. Sparsity problem
n_users_train  = len(user_ids)
n_movies_train = len(movie_ids)
n_ratings      = train_matrix.nnz
sparsity       = 1 - n_ratings / (
    n_users_train * n_movies_train)

print(f"\n1. SPARSITY PROBLEM")
print(f"   Matrix size  : {n_users_train:,} × "
      f"{n_movies_train:,}")
print(f"   Filled cells : {n_ratings:,}")
print(f"   Sparsity     : {sparsity*100:.4f}%")
print(f"   → Most user pairs share 0 rated movies")
print(f"   → Similarity scores are unreliable")

# 2. Cold start
cold_start_users = len(test) - len(test_clean)
print(f"\n2. COLD START PROBLEM")
print(f"   Test ratings removed : {cold_start_users:,}")
print(f"   % of test set        : "
      f"{cold_start_users/len(test)*100:.1f}%")
print(f"   → New users get zero recommendations")
print(f"   → New movies never get recommended")

# 3. Popularity bias
rec_movie_counts = defaultdict(int)
sample_users = train['userId'].unique()[:100]

for uid in sample_users:
    recs = item_cf.recommend(int(uid), 10)
    for mid, _ in recs:
        rec_movie_counts[mid] += 1

top_recommended = sorted(
    rec_movie_counts.items(),
    key=lambda x: x[1], reverse=True
)[:10]

print(f"\n3. POPULARITY BIAS")
print(f"   Top 10 most recommended movies:")
for mid, count in top_recommended[:5]:
    title = movies[movies['movieId'] == mid][
        'title'].values
    title = title[0] if len(title) > 0 else str(mid)
    print(f"   {title:<40} recommended {count}x")
print(f"   → Same popular movies recommended to everyone")
print(f"   → Long tail movies never recommended")

print(f"""
CONCLUSION FOR REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CF fails because:
  1. 99.8% sparsity → unreliable similarity scores
  2. Cold start → 0 recommendations for new users
  3. Popularity bias → top 1% movies dominate

This motivates:
  → SVD/ALS (Day 9) — handles sparsity better
  → GRank generative retrieval (Day 10)
  → CLIP embeddings (Day 12) — solves cold start
  → IPS debiasing (Week 4) — fixes popularity bias
""")

CF FAILURE ANALYSIS

1. SPARSITY PROBLEM
   Matrix size  : 547 × 7,356
   Filled cells : 80,003
   Sparsity     : 98.0117%
   → Most user pairs share 0 rated movies
   → Similarity scores are unreliable

2. COLD START PROBLEM
   Test ratings removed : 9,577
   % of test set        : 95.8%
   → New users get zero recommendations
   → New movies never get recommended

3. POPULARITY BIAS
   Top 10 most recommended movies:
   Pretty Woman                             recommended 48x
   Mrs. Doubtfire                           recommended 42x
   Speed                                    recommended 40x
   Se7en                                    recommended 34x
   Braveheart                               recommended 31x
   → Same popular movies recommended to everyone
   → Long tail movies never recommended

CONCLUSION FOR REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CF fails because:
  1. 99.8% sparsity → unreliable similarity scores
  2. Cold start → 0 recommendations for new us

In [14]:
# Save Models + Results

import joblib
import os

os.makedirs('../../models/checkpoints', exist_ok=True)

# Save models
joblib.dump(user_cf,
    '../../models/checkpoints/user_cf.joblib')
joblib.dump(item_cf,
    '../../models/checkpoints/item_cf.joblib')

# Save mappings
joblib.dump({
    'user2idx':  user2idx,
    'movie2idx': movie2idx,
    'idx2user':  idx2user,
    'idx2movie': idx2movie,
    'user_ids':  user_ids,
    'movie_ids': movie_ids,
}, '../../models/checkpoints/cf_mappings.joblib')

# Save results
results_all = {
    'user_cf': results_user_cf,
    'item_cf': results_item_cf,
}
with open(PROC + 'cf_results.json', 'w') as f:
    json.dump(results_all, f, indent=2)

print("✅ Models and results saved")
print(json.dumps(results_all, indent=2))

✅ Models and results saved
{
  "user_cf": {
    "model": "User-CF",
    "rmse": 1.0497,
    "mae": 0.7457,
    "precision@10": 0.0714,
    "recall@10": 0.0543,
    "ndcg@10": 0.0686,
    "n_predictions": 280,
    "n_users_eval": 7
  },
  "item_cf": {
    "model": "Item-CF",
    "rmse": 0.8904,
    "mae": 0.6438,
    "precision@10": 0.0286,
    "recall@10": 0.0248,
    "ndcg@10": 0.0367,
    "n_predictions": 416,
    "n_users_eval": 7
  }
}


In [15]:
## Write Unit Tests

# Write test file
test_code = '''
import sys
sys.path.insert(0, "../..")
import numpy as np
import scipy.sparse as sp
import pytest

# ── fixtures ──────────────────────────────────────
@pytest.fixture
def small_matrix():
    """Tiny 4-user x 5-movie matrix for testing"""
    data = np.array([
        [5, 4, 0, 0, 1],
        [4, 5, 0, 0, 1],
        [0, 0, 4, 5, 0],
        [0, 0, 5, 4, 0],
    ], dtype=float)
    return sp.csr_matrix(data)

@pytest.fixture
def mappings():
    user2idx  = {1:0, 2:1, 3:2, 4:3}
    movie2idx = {10:0, 20:1, 30:2, 40:3, 50:4}
    idx2movie = {v:k for k,v in movie2idx.items()}
    idx2user  = {v:k for k,v in user2idx.items()}
    return user2idx, movie2idx, idx2movie, idx2user

# ── import models ─────────────────────────────────
from notebooks.week2_retrieval\
     .collaborative_filtering_module import (
    UserCF, ItemCF,
    cosine_similarity_sparse,
    rmse, mae, precision_at_k,
    recall_at_k, ndcg_at_k
)

# ── similarity tests ──────────────────────────────
def test_cosine_similarity_shape(small_matrix):
    sim = cosine_similarity_sparse(small_matrix)
    assert sim.shape == (4, 4)

def test_cosine_similarity_diagonal(small_matrix):
    sim = cosine_similarity_sparse(small_matrix)
    np.testing.assert_array_almost_equal(
        np.diag(sim), np.ones(4), decimal=5)

def test_cosine_similarity_range(small_matrix):
    sim = cosine_similarity_sparse(small_matrix)
    assert sim.min() >= -1.0
    assert sim.max() <= 1.0 + 1e-6

# ── metric tests ──────────────────────────────────
def test_rmse_perfect():
    preds = [(3.0, 3.0), (4.0, 4.0), (5.0, 5.0)]
    assert rmse(preds) == 0.0

def test_mae_perfect():
    preds = [(3.0, 3.0), (4.0, 4.0)]
    assert mae(preds) == 0.0

def test_precision_at_k_perfect():
    recs     = [1, 2, 3, 4, 5]
    relevant = {1, 2, 3, 4, 5}
    assert precision_at_k(recs, relevant, 5) == 1.0

def test_precision_at_k_zero():
    recs     = [1, 2, 3]
    relevant = {4, 5, 6}
    assert precision_at_k(recs, relevant, 3) == 0.0

def test_ndcg_at_k_perfect():
    recs     = [1, 2, 3]
    relevant = {1, 2, 3}
    assert ndcg_at_k(recs, relevant, 3) == 1.0
'''

# Save test file
test_path = Path('../../tests/unit/test_cf.py')
test_path.parent.mkdir(parents=True, exist_ok=True)
test_path.write_text(test_code)

print("✅ Unit tests written to tests/unit/test_cf.py")
print("\nRun tests with:")
print("  cd /path/to/project && pytest tests/unit/test_cf.py -v")

✅ Unit tests written to tests/unit/test_cf.py

Run tests with:
  cd /path/to/project && pytest tests/unit/test_cf.py -v
